# Hold custom gesture model — Google MediaPipe Model Maker

This notebook trains `start`, `pause`, and required `none` classes from the ZIP exported by Hold Gesture Lab.

> MediaPipe Model Maker is deprecated/no longer actively maintained. Use this notebook as an experimental benchmark pipeline, not as a permanent runtime dependency.


In [ ]:
!python -m pip install --upgrade pip -q
!python -m pip install mediapipe-model-maker -q


In [ ]:
from google.colab import files
import os, shutil, zipfile, json
import tensorflow as tf
from mediapipe_model_maker import gesture_recognizer

print("TensorFlow", tf.__version__)


## Upload the dataset ZIP

Use the ZIP exported by `train-gestures.html`. The archive contains `hold-gesture-dataset/start`, `pause`, and `none`.


In [ ]:
uploaded = files.upload()
zip_name = next(iter(uploaded))
work_dir = "/content/hold-training"
shutil.rmtree(work_dir, ignore_errors=True)
os.makedirs(work_dir, exist_ok=True)
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(work_dir)

dataset_path = os.path.join(work_dir, "hold-gesture-dataset")
assert os.path.isdir(dataset_path), f"Dataset folder not found: {dataset_path}"
labels = sorted(d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d)))
print("labels:", labels)
assert "none" in labels, "Model Maker requires a label named none"
for label in labels:
    n = len([f for f in os.listdir(os.path.join(dataset_path, label)) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    print(f"{label}: {n} images")


## Load and split

Google's customization guide uses an 80% / 10% / 10% split. Images where the preprocessing hand detector cannot find a hand are omitted.


In [ ]:
data = gesture_recognizer.Dataset.from_folder(
    dirname=dataset_path,
    hparams=gesture_recognizer.HandDataPreprocessingParams()
)
train_data, rest_data = data.split(0.8)
validation_data, test_data = rest_data.split(0.5)
print("train:", train_data.size)
print("validation:", validation_data.size)
print("test:", test_data.size)


## Train with defaults

First run intentionally uses Model Maker defaults. Tune only after we have a baseline.


In [ ]:
export_dir = "/content/hold-exported-model"
shutil.rmtree(export_dir, ignore_errors=True)
hparams = gesture_recognizer.HParams(export_dir=export_dir)
options = gesture_recognizer.GestureRecognizerOptions(hparams=hparams)
model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options
)


## Evaluate


In [ ]:
loss, accuracy = model.evaluate(test_data, batch_size=1)
print(f"Test loss: {loss:.6f}")
print(f"Test accuracy: {accuracy:.4%}")


## Export `.task` and download

Do not overwrite the production gesture model yet. Rename this download to `hold-gestures-v1.task` and A/B test it first.


In [ ]:
model.export_model()
model_path = os.path.join(export_dir, "gesture_recognizer.task")
assert os.path.exists(model_path), model_path
print(model_path, os.path.getsize(model_path), "bytes")
files.download(model_path)
